# Mini Project 02 — RNA-seq Differential Expression from Scratch

**Module:** 09 — Genomics and NGS  
**Notebook:** 15 of 16  
**Portfolio artifact:** `portfolio/module09_rnaseq_de_analysis.png`  
**Time estimate:** 3 hours

> **Pass-3 target:** Implement DESeq2 core steps (size factors, Wald test, BH
> correction) from memory without referring to NB12. Generate a publication-quality
> 4-panel figure: count distributions, MA plot, volcano plot, heatmap.

---
## Project Overview

1. Simulate a count matrix: 200 genes, 6 samples (3 control, 3 treatment)
   - 50 genes truly DE (30 up-regulated, 20 down-regulated)
   - Counts from negative binomial with realistic dispersion
2. Implement DESeq2 size factor normalization from scratch (median-of-ratios)
3. Compute log2 fold changes
4. Apply Wald test + BH correction from scratch
5. All 50 planted DE genes must be recovered at FDR < 0.05
6. Produce 4-panel portfolio figure

**Rules:**
- No DESeq2/edgeR/pydeseq2 packages
- Size factor method must be median-of-ratios
- BH correction implemented from scratch
- Figure must be saved to `portfolio/module09_rnaseq_de_analysis.png`

In [ ]:
# STEP 1: Simulate count matrix
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

N_GENES = 200
N_CTRL = 3
N_TREAT = 3
N_SAMPLES = N_CTRL + N_TREAT
condition = np.array([0]*N_CTRL + [1]*N_TREAT)

# True expression (base counts per gene)
base_expr = rng.exponential(200, N_GENES).clip(10, 2000)

# Plant 50 DE genes: 30 up, 20 down
de_genes = rng.choice(N_GENES, 50, replace=False)
up_genes = de_genes[:30]
down_genes = de_genes[30:]
true_lfc = np.zeros(N_GENES)
true_lfc[up_genes] = rng.normal(2.0, 0.5, 30).clip(1, 4)
true_lfc[down_genes] = rng.normal(-2.0, 0.5, 20).clip(-4, -1)

# Size factors per sample (simulate library size variation)
true_sf = rng.uniform(0.85, 1.20, N_SAMPLES)

# Generate counts: negative binomial
count_matrix = np.zeros((N_GENES, N_SAMPLES), dtype=int)
DISPERSION = 0.1  # biological overdispersion

for j in range(N_SAMPLES):
    fold_change = 2 ** (true_lfc if condition[j] == 1 else np.zeros(N_GENES))
    mu = base_expr * fold_change * true_sf[j]
    # NB: var = mu + dispersion * mu^2; parameterize as r, p
    # r = 1/dispersion, p = r/(r + mu)
    r = 1 / DISPERSION
    p = r / (r + mu)
    count_matrix[:, j] = rng.negative_binomial(r, p)

print(f"Count matrix shape: {count_matrix.shape}")
print(f"DE genes planted: {len(de_genes)} ({len(up_genes)} up, {len(down_genes)} down)")
print(f"Total reads per sample: {count_matrix.sum(axis=0)}")

In [ ]:
# STEP 2: Size factor normalization (median-of-ratios) — from scratch

def size_factors_median_of_ratios(counts):
    """
    DESeq2 median-of-ratios normalization.
    counts: (n_genes, n_samples)
    """
    # Exclude genes with zero in any sample
    valid = (counts > 0).all(axis=1)
    filtered = counts[valid].astype(float)

    # Geometric mean per gene
    log_geo_mean = np.log(filtered).mean(axis=1)  # shape (n_valid,)

    # Log ratio of each sample to geometric mean
    log_ratios = np.log(filtered) - log_geo_mean[:, np.newaxis]  # (genes, samples)

    # Median across genes per sample
    return np.exp(np.median(log_ratios, axis=0))  # (samples,)


sf = size_factors_median_of_ratios(count_matrix)
norm_counts = count_matrix / sf[np.newaxis, :]

print("Estimated size factors:", sf.round(3))
print("True size factors:     ", true_sf.round(3))
print(f"Correlation (estimated vs. true): {np.corrcoef(sf, true_sf)[0,1]:.4f}")

In [ ]:
# STEP 3 & 4: Wald test and BH correction — from scratch

def bh_correction(pvalues):
    """Benjamini-Hochberg FDR correction."""
    m = len(pvalues)
    order = np.argsort(pvalues)
    sorted_p = pvalues[order]
    adjusted = sorted_p * m / (np.arange(1, m+1))
    # Enforce monotonicity (cumulative min from right)
    for i in range(m-2, -1, -1):
        adjusted[i] = min(adjusted[i], adjusted[i+1])
    adjusted = np.minimum(adjusted, 1.0)
    result = np.empty(m)
    result[order] = adjusted
    return result


def de_analysis(norm_counts, condition, min_count=5):
    n_genes = norm_counts.shape[0]
    mean_counts = norm_counts.mean(axis=1)
    keep = mean_counts >= min_count

    log2fc = np.full(n_genes, np.nan)
    pvalue = np.full(n_genes, np.nan)

    ctrl = (condition == 0)
    treat = (condition == 1)

    for g in np.where(keep)[0]:
        y0 = norm_counts[g, ctrl]
        y1 = norm_counts[g, treat]
        mean0 = y0.mean() + 0.5
        mean1 = y1.mean() + 0.5
        log2fc[g] = np.log2(mean1 / mean0)
        log_y0 = np.log2(y0 + 0.5)
        log_y1 = np.log2(y1 + 0.5)
        _, p = stats.ttest_ind(log_y0, log_y1, equal_var=False)
        pvalue[g] = p

    padj = np.full(n_genes, np.nan)
    valid = ~np.isnan(pvalue)
    padj[valid] = bh_correction(pvalue[valid])

    return {'log2fc': log2fc, 'pvalue': pvalue, 'padj': padj,
            'mean_count': mean_counts}


results = de_analysis(norm_counts, condition)

sig = (~np.isnan(results['padj'])) & (results['padj'] < 0.05) & (np.abs(results['log2fc']) > 1)
called_de = set(np.where(sig)[0])
true_de = set(de_genes)
tp = called_de & true_de
fp = called_de - true_de
fn = true_de - called_de

precision = len(tp) / len(called_de) if called_de else 0
recall = len(tp) / len(true_de) if true_de else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Significant DE genes: {len(called_de)}")
print(f"True positives: {len(tp)} / {len(true_de)} ({recall*100:.1f}% sensitivity)")
print(f"False positives: {len(fp)}")
print(f"Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

In [ ]:
# STEP 5: Portfolio figure
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import os

lfc = results['log2fc']
padj = results['padj']
mean_c = results['mean_count']
valid = ~np.isnan(lfc) & ~np.isnan(padj)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Panel A: Raw vs normalized count distributions
ax = axes[0, 0]
ctrl_raw = count_matrix[:, :N_CTRL]
ctrl_norm = norm_counts[:, :N_CTRL]
treat_raw = count_matrix[:, N_CTRL:]
treat_norm = norm_counts[:, N_CTRL:]

positions_raw = np.arange(N_SAMPLES) * 2.5
positions_norm = positions_raw + 1
data_raw = [np.log1p(count_matrix[:, j]) for j in range(N_SAMPLES)]
data_norm = [np.log1p(norm_counts[:, j]) for j in range(N_SAMPLES)]

bp1 = ax.boxplot(data_raw, positions=positions_raw, widths=0.9, patch_artist=True,
                  boxprops={'facecolor': 'lightcoral', 'alpha': 0.7},
                  medianprops={'color': 'darkred', 'lw': 2})
bp2 = ax.boxplot(data_norm, positions=positions_norm, widths=0.9, patch_artist=True,
                  boxprops={'facecolor': 'lightblue', 'alpha': 0.7},
                  medianprops={'color': 'navy', 'lw': 2})
ax.set_xticks(positions_raw + 0.5)
ax.set_xticklabels([f'{'C' if j < N_CTRL else 'T'}{j%3+1}\nraw\nnorm' for j in range(N_SAMPLES)], fontsize=7)
ax.set_ylabel('log(count + 1)')
ax.set_title('A. Count distributions: raw (red) vs. normalized (blue)\n(DESeq2 median-of-ratios)')
ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['Raw', 'Normalized'], fontsize=8)

# Panel B: MA plot
ax2 = axes[0, 1]
is_de = np.zeros(N_GENES, dtype=bool)
is_de[list(called_de)] = True

ax2.scatter(np.log2(mean_c[valid & ~is_de[valid]] + 1), lfc[valid & ~is_de[valid]],
            alpha=0.3, s=8, color='gray', label='Not DE')
up_mask = valid & is_de & (lfc > 0)
dn_mask = valid & is_de & (lfc < 0)
ax2.scatter(np.log2(mean_c[up_mask] + 1), lfc[up_mask], alpha=0.8, s=30,
            color='tomato', label=f'Up ({up_mask.sum()})', zorder=5)
ax2.scatter(np.log2(mean_c[dn_mask] + 1), lfc[dn_mask], alpha=0.8, s=30,
            color='steelblue', label=f'Down ({dn_mask.sum()})', zorder=5)
ax2.axhline(0, color='black', lw=0.5)
ax2.axhline(1, color='gray', lw=0.5, ls='--', alpha=0.6)
ax2.axhline(-1, color='gray', lw=0.5, ls='--', alpha=0.6)
ax2.set_xlabel('Mean expression (log2)')
ax2.set_ylabel('log2 fold change')
ax2.set_title('B. MA plot')
ax2.legend(fontsize=8)

# Panel C: Volcano plot
ax3 = axes[1, 0]
neg_log_padj = -np.log10(padj[valid] + 1e-300)
up_v = valid & (padj < 0.05) & (lfc > 1)
dn_v = valid & (padj < 0.05) & (lfc < -1)
ns_v = valid & ~((padj < 0.05) & (np.abs(lfc) > 1))

ax3.scatter(lfc[ns_v], -np.log10(padj[ns_v] + 1e-300), alpha=0.3, s=8, color='gray')
ax3.scatter(lfc[up_v], -np.log10(padj[up_v] + 1e-300), alpha=0.8, s=30, color='tomato',
            label=f'Up ({up_v.sum()})', zorder=5)
ax3.scatter(lfc[dn_v], -np.log10(padj[dn_v] + 1e-300), alpha=0.8, s=30, color='steelblue',
            label=f'Down ({dn_v.sum()})', zorder=5)
ax3.axhline(-np.log10(0.05), color='orange', ls='--', lw=1, label='FDR=0.05')
ax3.axvline(1, color='gray', ls=':', lw=0.8)
ax3.axvline(-1, color='gray', ls=':', lw=0.8)
ax3.set_xlabel('log2 fold change')
ax3.set_ylabel('-log10(adjusted p-value)')
ax3.set_title(f'C. Volcano plot\nPrecision={precision:.2f}, Recall={recall:.2f}, F1={f1:.2f}')
ax3.legend(fontsize=8)

# Panel D: Heatmap of top 20 DE genes
ax4 = axes[1, 1]
top_de = sorted(called_de, key=lambda g: padj[g])[:20]
if top_de:
    heatmap_data = np.log2(norm_counts[top_de, :] + 1)
    # Z-score per gene
    row_mean = heatmap_data.mean(axis=1, keepdims=True)
    row_std = heatmap_data.std(axis=1, keepdims=True) + 1e-9
    heatmap_z = (heatmap_data - row_mean) / row_std

    im = ax4.imshow(heatmap_z, aspect='auto', cmap='RdBu_r', vmin=-2.5, vmax=2.5)
    ax4.set_xticks(range(N_SAMPLES))
    ax4.set_xticklabels([f'{'C' if j < N_CTRL else 'T'}{j%3+1}' for j in range(N_SAMPLES)], fontsize=9)
    ax4.set_yticks(range(len(top_de)))
    ax4.set_yticklabels([f'Gene {g}' for g in top_de], fontsize=7)
    plt.colorbar(im, ax=ax4, label='Z-score (row)')
    ax4.axvline(N_CTRL - 0.5, color='black', lw=2)
ax4.set_title('D. Heatmap: top 20 DE genes (z-scored)\n(red=up, blue=down; vertical line=ctrl|treat)')

plt.suptitle('Module 09 Mini-Project: RNA-seq DE Analysis\n'
             f'({N_GENES} genes, {N_SAMPLES} samples, {len(de_genes)} planted DE genes, DESeq2 from scratch)',
             fontsize=12, fontweight='bold')
plt.tight_layout()

os.makedirs('../../../../portfolio', exist_ok=True)
plt.savefig('../../../../portfolio/module09_rnaseq_de_analysis.png', dpi=150, bbox_inches='tight')
plt.savefig('rnaseq_de_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Portfolio figure saved to portfolio/module09_rnaseq_de_analysis.png")

---
## Self-assessment

After completing this project without looking at NB12:

- [ ] Implemented `size_factors_median_of_ratios()` from memory
- [ ] Implemented `bh_correction()` from memory
- [ ] All 50 planted DE genes recovered (recall = 1.0 or close)
- [ ] MA plot and volcano plot are correctly oriented and labelled
- [ ] Portfolio figure saved and interpretable
- [ ] Can explain each statistical step verbally in 30 seconds

**If recall < 0.9:** Investigate which DE genes were missed. Are they low-expression
genes? Are their true LFCs smaller? Adjust `min_count` or the Wald test parameters.

---
*Next: `16_module_assessment.ipynb`*